# AML 01 — Data preparation and shared Wi-Fi features

This notebook inspects the supplied raw partitions, defines one reproducible AML development split, builds model-ready Wi-Fi features without targets or identifiers in the predictor matrix, and exports reusable tables under `data/processed/`.

**Team contract:** split `trainingData.csv` by `(BUILDINGID, FLOOR, SPACEID)`, with 20% of distinct spaces assigned to development using seed `42`. Use `FEATURE_COLUMNS` below as the predictor allowlist. Keep `validationData.csv` held out for final evaluation. The signal mapping is fixed: raw sentinel `100` becomes `-110` dBm plus a per-AP seen indicator; no transformation is fitted on validation data.


In [1]:
from pathlib import Path
import platform
import re
import sys
import numpy as np
import pandas as pd

print(f"Python: {platform.python_version()} ({sys.executable})")
print(f"NumPy: {np.__version__}; pandas: {pd.__version__}")

def find_project_root(start=None):
    candidate = Path.cwd() if start is None else Path(start)
    for path in (candidate, *candidate.parents):
        if (path / "data/raw/trainingData.csv").is_file() and (path / "Machine Learning Pipeline").is_dir():
            return path
    raise FileNotFoundError("Could not locate repository root")

PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TRAIN_PATH = RAW_DIR / "trainingData.csv"
VALIDATION_PATH = RAW_DIR / "validationData.csv"
assert TRAIN_PATH.is_file() and VALIDATION_PATH.is_file(), "Both raw CSV inputs are required"
print("Project root:", PROJECT_ROOT)
print("Inputs:", TRAIN_PATH.relative_to(PROJECT_ROOT), VALIDATION_PATH.relative_to(PROJECT_ROOT))


Python: 3.14.2 (/opt/homebrew/Cellar/jupyterlab/4.5.3/libexec/bin/python)
NumPy: 2.5.2; pandas: 3.0.5
Project root: /Users/bhawana./Documents/Codex/2026-10-21/files-pasted-by-the-user-you/Signal2Space
Inputs: data/raw/trainingData.csv data/raw/validationData.csv


## 1. Load and validate the schema

WAP columns are discovered from the verified names using `WAP` plus digits. Targets are building, floor, longitude, and latitude. Space, relative position, user, phone, and timestamp are retained for traceability and robustness analysis, never used as predictors by default.


In [2]:
train_raw = pd.read_csv(TRAIN_PATH)
validation_raw = pd.read_csv(VALIDATION_PATH)
EXPECTED_COLUMNS = train_raw.columns.tolist()
AP_COLUMNS = sorted([c for c in EXPECTED_COLUMNS if re.fullmatch(r"WAP\d+", str(c))], key=lambda c: int(c[3:]))
TARGET_COLUMNS = ["BUILDINGID", "FLOOR", "LONGITUDE", "LATITUDE"]
METADATA_COLUMNS = ["SPACEID", "RELATIVEPOSITION", "USERID", "PHONEID", "TIMESTAMP"]

def validate_schema(frame, expected_columns, partition_name):
    if frame.columns.tolist() != expected_columns:
        raise ValueError(f"{partition_name} schema/order differs from trainingData.csv")
    missing = frame.isna().sum()
    if missing.any():
        raise ValueError(f"Unexpected blank/NaN cells in {partition_name}: {missing[missing > 0].to_dict()}")
    absent = [c for c in TARGET_COLUMNS + METADATA_COLUMNS if c not in frame.columns]
    if absent:
        raise ValueError(f"{partition_name} lacks fields: {absent}")

validate_schema(train_raw, EXPECTED_COLUMNS, "trainingData.csv")
validate_schema(validation_raw, EXPECTED_COLUMNS, "validationData.csv")
assert len(AP_COLUMNS) == 520
assert AP_COLUMNS == [f"WAP{i:03d}" for i in range(1, len(AP_COLUMNS) + 1)]
train_raw = train_raw.copy()
validation_raw = validation_raw.copy()
train_raw.insert(0, "ROW_ID", np.arange(len(train_raw), dtype=np.int32))
validation_raw.insert(0, "ROW_ID", np.arange(len(validation_raw), dtype=np.int32))
print("Rows × columns:", train_raw.shape, validation_raw.shape)
print("Raw dtype counts:", train_raw[EXPECTED_COLUMNS].dtypes.astype(str).value_counts().to_dict())
print("Target dtypes:", train_raw[TARGET_COLUMNS].dtypes.astype(str).to_dict())
print("Metadata dtypes:", train_raw[METADATA_COLUMNS].dtypes.astype(str).to_dict())
print("Building labels:", sorted(train_raw.BUILDINGID.unique().tolist()))
print("Floor labels:", sorted(train_raw.FLOOR.unique().tolist()))
print("Target fields available in validation:", {c: int(validation_raw[c].notna().sum()) for c in TARGET_COLUMNS})


Rows × columns: (19937, 530) (1111, 530)
Raw dtype counts: {'int64': 527, 'float64': 2}
Target dtypes: {'BUILDINGID': 'int64', 'FLOOR': 'int64', 'LONGITUDE': 'float64', 'LATITUDE': 'float64'}
Metadata dtypes: {'SPACEID': 'int64', 'RELATIVEPOSITION': 'int64', 'USERID': 'int64', 'PHONEID': 'int64', 'TIMESTAMP': 'int64'}
Building labels: [0, 1, 2]
Floor labels: [0, 1, 2, 3, 4]
Target fields available in validation: {'BUILDINGID': 1111, 'FLOOR': 1111, 'LONGITUDE': 1111, 'LATITUDE': 1111}


## 2. Data-quality checks and missing-signal convention

Blank fields are distinguished from the WAP sentinel. Exact duplicate rows are reported but retained because the repository provides no basis for deciding they are accidental; dropping them would change sample weights. `ROW_ID` records the stable row position in its source file.


In [3]:
train_duplicates = int(train_raw[EXPECTED_COLUMNS].duplicated().sum())
validation_duplicates = int(validation_raw[EXPECTED_COLUMNS].duplicated().sum())
MISSING_SIGNAL = 100
ABSENT_RSSI_DBM = -110.0
train_signals = train_raw[AP_COLUMNS].to_numpy(dtype=np.int16, copy=True)
validation_signals = validation_raw[AP_COLUMNS].to_numpy(dtype=np.int16, copy=True)
train_is_missing = train_signals == MISSING_SIGNAL
train_observed = ~train_is_missing
assert train_is_missing.any() and train_observed.any()
assert np.all(train_signals[train_observed] <= 0), "Unexpected nonnegative training signal besides 100"
assert np.all(validation_signals[validation_signals != MISSING_SIGNAL] <= 0), "Unexpected nonnegative validation signal"
assert ABSENT_RSSI_DBM < float(train_signals[train_observed].min())
print("Exact duplicate raw rows retained:", {"training": train_duplicates, "validation": validation_duplicates})
print("Blank cell counts:", {"training": int(train_raw[EXPECTED_COLUMNS].isna().sum().sum()), "validation": int(validation_raw[EXPECTED_COLUMNS].isna().sum().sum())})
print("Training WAP min/max including sentinel:", (int(train_signals.min()), int(train_signals.max())))
print("Observed training RSSI min/max dBm:", (int(train_signals[train_observed].min()), int(train_signals[train_observed].max())))
print("Training sentinel count/fraction:", int(train_is_missing.sum()), round(float(train_is_missing.mean()), 6))
print("Detected APs per train row min/median/max:", int(train_observed.sum(1).min()), float(np.median(train_observed.sum(1))), int(train_observed.sum(1).max()))
print("Detected APs per validation row min/median/max:", int((validation_signals != MISSING_SIGNAL).sum(1).min()), float(np.median((validation_signals != MISSING_SIGNAL).sum(1))), int((validation_signals != MISSING_SIGNAL).sum(1).max()))
print("Held-out placeholder metadata:", {c: validation_raw[c].value_counts().to_dict() for c in ["SPACEID", "RELATIVEPOSITION", "USERID"]})
# Compare exact supplied records without combining the partitions for fitting.
train_hashes = pd.util.hash_pandas_object(train_raw[EXPECTED_COLUMNS], index=False).to_numpy()
validation_hashes = pd.util.hash_pandas_object(validation_raw[EXPECTED_COLUMNS], index=False).to_numpy()
exact_partition_overlap = int(np.intersect1d(train_hashes, validation_hashes).size)
print("Exact-row hash overlap between supplied files:", exact_partition_overlap)
assert exact_partition_overlap == 0


Exact duplicate raw rows retained: {'training': 637, 'validation': 0}
Blank cell counts: {'training': 0, 'validation': 0}
Training WAP min/max including sentinel: (-104, 100)
Observed training RSSI min/max dBm: (-104, 0)
Training sentinel count/fraction: 10008477 0.965395
Detected APs per train row min/median/max: 0 17.0 51
Detected APs per validation row min/median/max: 1 15.0 35
Held-out placeholder metadata: {'SPACEID': {0: 1111}, 'RELATIVEPOSITION': {0: 1111}, 'USERID': {0: 1111}}


Exact-row hash overlap between supplied files: 0


## 3. Reproducible grouped development split

The repository has no partition manifest, but the files have identical schemas and no exact raw records in common; validation metadata fields `SPACEID`, `RELATIVEPOSITION`, and `USERID` are zero placeholders. Treat `validationData.csv` as held out.

Split only the supplied training partition by `(BUILDINGID, FLOOR, SPACEID)`. The seeded group permutation mirrors the group selection used by `GroupShuffleSplit(test_size=0.20, random_state=42)`: sort distinct group keys, permute them using NumPy's legacy `RandomState(42)`, and assign the first `ceil(20% × group_count)` groups to internal development. Every row from one physical-space group stays in one side.


In [4]:
GROUP_COLUMNS = ["BUILDINGID", "FLOOR", "SPACEID"]
GROUP_TEST_SIZE = 0.20
GROUP_RANDOM_SEED = 42
group_keys = train_raw[GROUP_COLUMNS].astype(str).agg("_".join, axis=1)
unique_groups, group_codes = np.unique(group_keys.to_numpy(), return_inverse=True)
permutation = np.random.RandomState(GROUP_RANDOM_SEED).permutation(len(unique_groups))
n_dev_groups = int(np.ceil(GROUP_TEST_SIZE * len(unique_groups)))
dev_group_codes = permutation[:n_dev_groups]
dev_mask = np.isin(group_codes, dev_group_codes)
internal_train_positions = np.flatnonzero(~dev_mask)
internal_validation_positions = np.flatnonzero(dev_mask)
internal_train_raw = train_raw.iloc[internal_train_positions].copy()
internal_validation_raw = train_raw.iloc[internal_validation_positions].copy()
assert set(group_keys.iloc[internal_train_positions]).isdisjoint(set(group_keys.iloc[internal_validation_positions]))
assert len(internal_train_raw) + len(internal_validation_raw) == len(train_raw)
print("Distinct physical spaces / development groups:", len(unique_groups), n_dev_groups)
print("Internal train/dev rows:", len(internal_train_raw), len(internal_validation_raw))
print("Building counts (train/dev):", internal_train_raw.BUILDINGID.value_counts().sort_index().to_dict(), internal_validation_raw.BUILDINGID.value_counts().sort_index().to_dict())
print("Floor counts (train/dev):", internal_train_raw.FLOOR.value_counts().sort_index().to_dict(), internal_validation_raw.FLOOR.value_counts().sort_index().to_dict())


Distinct physical spaces / development groups: 735 147
Internal train/dev rows: 16085 3852
Building counts (train/dev): {0: 4177, 1: 4102, 2: 7806} {0: 1072, 1: 1094, 2: 1686}
Floor counts (train/dev): {0: 3569, 1: 3892, 2: 3592, 3: 4170, 4: 862} {0: 800, 1: 1110, 2: 824, 3: 878, 4: 240}


## 4. Signal encoding and shared engineered features

For each discovered WAP, output `RSSI__WAPnnn` (`float32`, observed dBm or `-110` when missing) and `SEEN__WAPnnn` (`uint8`, 1 for a detected AP, otherwise 0). Six summaries are `n_heard` (int16), `max_rssi`, `mean_rssi`, `std_rssi` (float32, over observed APs), `n_strong` (int16, observed readings above −70 dBm), and `top1_ap` (int16, zero-based AP position of strongest observed signal, or `-1` if none). Empty-signal rows use `-110` for max/mean, 0 for standard deviation, and -1 for top AP; `n_heard` and the seen mask make these cases explicit.

No scaling is applied: raw dBm is interpretable and appropriate for the tree-based AML models. The fixed mapping is applied to all partitions. The fill value is below all observed training values. No validation-derived value is used to fit preprocessing.


In [5]:
def encode_wifi(frame, ap_columns=AP_COLUMNS, missing_signal=MISSING_SIGNAL, absent_dbm=ABSENT_RSSI_DBM):
    raw = frame[ap_columns].to_numpy(dtype=np.float32, copy=True)
    seen_array = raw != missing_signal
    rssi = pd.DataFrame(np.where(seen_array, raw, absent_dbm).astype(np.float32), columns=[f"RSSI__{ap}" for ap in ap_columns], index=frame.index)
    seen = pd.DataFrame(seen_array.astype(np.uint8), columns=[f"SEEN__{ap}" for ap in ap_columns], index=frame.index)
    return rssi, seen

def make_features(rssi, seen):
    d = rssi.to_numpy(dtype=np.float32, copy=False)
    m = seen.to_numpy(dtype=bool, copy=False)
    if d.shape != m.shape:
        raise ValueError("RSSI and seen-mask matrices must have the same shape")
    count = m.sum(axis=1).astype(np.int16)
    has_signal = count > 0
    observed = np.where(m, d, np.nan)
    maximum = np.full(len(d), ABSENT_RSSI_DBM, dtype=np.float32)
    mean = np.full(len(d), ABSENT_RSSI_DBM, dtype=np.float32)
    std = np.zeros(len(d), dtype=np.float32)
    top1 = np.full(len(d), -1, dtype=np.int16)
    if has_signal.any():
        maximum[has_signal] = np.nanmax(observed[has_signal], axis=1)
        mean[has_signal] = np.nanmean(observed[has_signal], axis=1)
        std[has_signal] = np.nanstd(observed[has_signal], axis=1)
        top1[has_signal] = np.argmax(d[has_signal], axis=1).astype(np.int16)
    strong = ((d > -70.0) & m).sum(axis=1).astype(np.int16)
    return pd.DataFrame({"n_heard": count, "max_rssi": maximum, "mean_rssi": mean, "std_rssi": std, "n_strong": strong, "top1_ap": top1}, index=rssi.index)

FEATURE_COLUMNS = ([f"RSSI__{ap}" for ap in AP_COLUMNS]
                   + [f"SEEN__{ap}" for ap in AP_COLUMNS]
                   + ["n_heard", "max_rssi", "mean_rssi", "std_rssi", "n_strong", "top1_ap"])
OUTPUT_METADATA_COLUMNS = ["ROW_ID", "SOURCE_PARTITION"] + METADATA_COLUMNS
OUTPUT_COLUMNS = OUTPUT_METADATA_COLUMNS + TARGET_COLUMNS + FEATURE_COLUMNS
assert len(FEATURE_COLUMNS) == 2 * len(AP_COLUMNS) + 6
assert not (set(FEATURE_COLUMNS) & set(TARGET_COLUMNS + OUTPUT_METADATA_COLUMNS))
# Smoke check: an all-missing row remains finite and does not claim WAP001 as strongest.
_smoke_raw = pd.DataFrame({ap: [MISSING_SIGNAL] for ap in AP_COLUMNS})
_smoke_rssi, _smoke_seen = encode_wifi(_smoke_raw)
_smoke_features = make_features(_smoke_rssi, _smoke_seen)
assert _smoke_features.loc[0, "n_heard"] == 0 and _smoke_features.loc[0, "top1_ap"] == -1
assert np.isfinite(_smoke_features.to_numpy(dtype=float)).all()
print("Predictor feature count:", len(FEATURE_COLUMNS), "(520 RSSI + 520 seen indicators + 6 summaries)")


Predictor feature count: 1046 (520 RSSI + 520 seen indicators + 6 summaries)


## 5. Prepare, validate, and export team artifacts

Each compressed CSV keeps `ROW_ID`, source partition, collection metadata, target labels, then the shared predictors. Use only `FEATURE_COLUMNS` for model input. Outputs can be read with `pd.read_csv` and regenerated by rerunning this notebook from the repository root.


In [6]:
def prepare_split(frame, source_partition):
    rssi, seen = encode_wifi(frame)
    summaries = make_features(rssi, seen)
    retained = frame[["ROW_ID"] + METADATA_COLUMNS + TARGET_COLUMNS].copy()
    retained.insert(1, "SOURCE_PARTITION", source_partition)
    prepared = pd.concat([retained, rssi, seen, summaries], axis=1)[OUTPUT_COLUMNS]
    if prepared[FEATURE_COLUMNS].isna().any().any():
        raise AssertionError("Prepared predictors contain missing values")
    if not np.isfinite(prepared[FEATURE_COLUMNS].to_numpy(dtype=np.float64)).all():
        raise AssertionError("Prepared predictors contain non-finite values")
    return prepared

internal_train = prepare_split(internal_train_raw, "trainingData")
internal_validation = prepare_split(internal_validation_raw, "trainingData")
heldout_validation = prepare_split(validation_raw, "validationData")
assert len(internal_train) == len(internal_train_raw)
assert len(internal_validation) == len(internal_validation_raw)
assert len(heldout_validation) == len(validation_raw)
assert internal_train.ROW_ID.is_unique and internal_validation.ROW_ID.is_unique and heldout_validation.ROW_ID.is_unique
assert set(internal_train.ROW_ID).isdisjoint(set(internal_validation.ROW_ID))
assert set(internal_train.ROW_ID) | set(internal_validation.ROW_ID) == set(train_raw.ROW_ID)
assert all(c in internal_train and c in heldout_validation for c in TARGET_COLUMNS + OUTPUT_METADATA_COLUMNS)
print("Prepared table shapes:", {"internal_train": internal_train.shape, "internal_validation": internal_validation.shape, "heldout_validation": heldout_validation.shape})
print("Example predictor dtypes:", {"RSSI": str(internal_train["RSSI__WAP001"].dtype), "SEEN": str(internal_train["SEEN__WAP001"].dtype), "n_heard": str(internal_train["n_heard"].dtype), "top1_ap": str(internal_train["top1_ap"].dtype)})

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS = {
    "aml_internal_train_features.csv.gz": internal_train,
    "aml_internal_validation_features.csv.gz": internal_validation,
    "aml_heldout_validation_features.csv.gz": heldout_validation,
}
for filename, frame in ARTIFACTS.items():
    path = PROCESSED_DIR / filename
    frame.to_csv(path, index=False, compression="gzip")
    reread = pd.read_csv(path)
    assert reread.shape == frame.shape and reread.columns.tolist() == OUTPUT_COLUMNS
    assert reread[FEATURE_COLUMNS].notna().all().all()
    print(filename, "rows:", len(reread), "columns:", len(reread.columns), "bytes:", path.stat().st_size)
print("Shared split:", GROUP_COLUMNS, "| development fraction:", GROUP_TEST_SIZE, "| seed:", GROUP_RANDOM_SEED)
print("Targets:", TARGET_COLUMNS)
print("Metadata retained but excluded from predictors:", OUTPUT_METADATA_COLUMNS)
print("No scaling fitted; validation used only the fixed sentinel mapping.")


Prepared table shapes: {'internal_train': (16085, 1057), 'internal_validation': (3852, 1057), 'heldout_validation': (1111, 1057)}
Example predictor dtypes: {'RSSI': 'float32', 'SEEN': 'uint8', 'n_heard': 'int16', 'top1_ap': 'int16'}


aml_internal_train_features.csv.gz rows: 16085 columns: 1057 bytes: 1815613


aml_internal_validation_features.csv.gz rows: 3852 columns: 1057 bytes: 450207


aml_heldout_validation_features.csv.gz rows: 1111 columns: 1057 bytes: 131634
Shared split: ['BUILDINGID', 'FLOOR', 'SPACEID'] | development fraction: 0.2 | seed: 42
Targets: ['BUILDINGID', 'FLOOR', 'LONGITUDE', 'LATITUDE']
Metadata retained but excluded from predictors: ['ROW_ID', 'SOURCE_PARTITION', 'SPACEID', 'RELATIVEPOSITION', 'USERID', 'PHONEID', 'TIMESTAMP']
No scaling fitted; validation used only the fixed sentinel mapping.


## Handoff notes and limitations

- Read `data/processed/aml_internal_train_features.csv.gz`, `aml_internal_validation_features.csv.gz`, and `aml_heldout_validation_features.csv.gz`. Each has the same schema: 7 traceability/metadata columns, 4 targets, and 1,046 predictors.
- The supplied validation file has available building/floor/coordinate labels, but `SPACEID`, `RELATIVEPOSITION`, and `USERID` are all zero placeholders. Preserve these fields for lineage; exclude them and all other metadata from predictors.
- Keep all 637 exact duplicate training records. The repository has no collection documentation to determine whether they are repeated measurements or accidental duplicate exports.
- Existing `ML_01_features.py` contains only six summaries and uses different behavior for a row with no detected AP. This notebook adds the per-AP signal/mask contract and handles the all-missing case explicitly. The script was not edited; members should consume these artifacts or align that helper before reuse.
- The repository does not document coordinate units/projection or additional semantics of `SPACEID`, `RELATIVEPOSITION`, `USERID`, `PHONEID`, or `TIMESTAMP`; values are preserved without reinterpretation.
